In [ ]:
!pip install anthropic chromadb python-docx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 923.9/923.9 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60

In [ ]:
import os
import anthropic
import chromadb
import docx
from google.colab import drive

drive.mount('/content/drive')

docs_folder = '/content/drive/MyDrive/HR-AI-Colabs-github-portfolio/onboarding-concierge/docs'

def extract_text_from_docx(filepath):
    doc = docx.Document(filepath)
    return "\n".join([para.text for para in doc.paragraphs if para.text.strip()])

# Load all docs
all_chunks = []
all_metadata = []

for filename in os.listdir(docs_folder):
    if filename.endswith('.docx'):
        filepath = os.path.join(docs_folder, filename)
        text = extract_text_from_docx(filepath)
        words = text.split()
        for i in range(0, len(words), 200):
            chunk = " ".join(words[i:i+200])
            if chunk:
                all_chunks.append(chunk)
                all_metadata.append({"source": filename})

print(f"Total chunks: {len(all_chunks)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total chunks: 12


In [6]:
# Load into ChromaDB
client = chromadb.Client()
collection = client.create_collection("onboarding")

for i in range(0, len(all_chunks), 50):
    batch = all_chunks[i:i+50]
    meta = all_metadata[i:i+50]
    ids = [str(j) for j in range(i, i+len(batch))]
    collection.add(documents=batch, metadatas=meta, ids=ids)

print(f"✓ {collection.count()} chunks loaded")

# Set up Claude
claude = anthropic.Anthropic(api_key="PASTE YOUR API KEY HERE")

def ask_onboarding_bot(question):
    results = collection.query(query_texts=[question], n_results=4)
    context = ""
    sources = []
    for doc, metadata in zip(results["documents"][0], results["metadatas"][0]):
        context += f"\n---\n{doc}"
        sources.append(metadata["source"])

    response = claude.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=800,
        messages=[{
            "role": "user",
            "content": f"You are a friendly onboarding assistant for Xuenix Entertainment, a Singapore tarot and oracle card company. Answer the new hire's question using only the context provided. Be warm and welcoming. If the answer is not in the context, say so and suggest they contact hr@xuenix.sg. Context: {context} Question: {question}"
        }]
    )
    print(f"\n{response.content[0].text}")
    print(f"\nSource: {list(set(sources))}")

# Test
ask_onboarding_bot("When do I get my laptop?")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 81.0MiB/s]


✓ 12 chunks loaded

Welcome to Xuenix! 🎉

Great question! Your laptop will be ready for you on your **first day (Monday) at 10:00am**. Your manager will hand it to you during the equipment collection phase. You'll also receive your charger, accessories, and a security token (YubiKey) at the same time.

Just remember to sign the asset acknowledgement form when you collect everything!

Is there anything else you'd like to know about your first day setup?

Source: ['06-new-hire-faq.docx', '05-first-week-schedule.docx', '02-it-setup-guide.docx']


In [7]:
questions = [
    "What is my annual leave entitlement?",
    "How do I claim medical expenses?",
    "Who do I contact for IT support?",
    "What should I do if I feel sick on a workday?"
]

for q in questions:
    print(f"\n{'='*50}")
    print(f"Q: {q}")
    ask_onboarding_bot(q)


Q: What is my annual leave entitlement?

Welcome to Xuenix Entertainment! 🎉

Great question! Your annual leave entitlement depends on your years of service with us. For the specific details based on how long you've been with the company, please refer to the **Employee Handbook** — it has the full entitlement schedule.

In the meantime, just remember that when you're ready to apply for leave, you can do so through the **HRIS portal** at hr.xuenix.sg (Leave > Apply). Just give at least **3 working days notice** for single-day leave, or **2 weeks** if you're taking 5 or more consecutive days.

If you can't find the information in your handbook or have any other questions, feel free to reach out to **hr@xuenix.sg** and they'll get back to you within 2 business days.

Is there anything else I can help you with? 😊

Source: ['03-benefits-guide.docx', '06-new-hire-faq.docx']

Q: How do I claim medical expenses?

Hi there! 👋 Great question!

Here's how to claim your outpatient medical expenses